In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

Vector Store


In [24]:
from langchain_community.vectorstores import Chroma
import numpy as np


1. Sample Document

In [4]:
sample_docs = [
    """Machine learning (ML) and Generative AI (GenAI) are both subsets of Artificial Intelligence (AI), but they have distinct purposes. ML focuses on enabling systems to learn from data and make predictions or decisions, while GenAI aims to generate new, original content like text, images, or music. 
Machine Learning (ML):
Definition:
ML is a field of AI that focuses on developing algorithms that allow computers to learn from data without explicit programming. 
Key Idea:
ML algorithms analyze data, identify patterns, and make predictions or decisions based on that data. 
Examples:
Image recognition: Identifying objects in images. 
Fraud detection: Identifying fraudulent transactions. 
Recommendation systems: Suggesting products or content based on user preferences. 
Predictive maintenance: Predicting when equipment will need maintenance. """
]


In [6]:
import tempfile
temp_dir = tempfile.mkdtemp()


for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)
        

Document Loading

In [9]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader = DirectoryLoader(
    "data",
    glob= "*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"{documents[0].page_content[:200]}")


Loaded 1 documents
Machine learning (ML) and Generative AI (GenAI) are both subsets of Artificial Intelligence (AI), but they have distinct purposes. ML focuses on enabling systems to learn from data and make prediction


Document Splitting

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)
chunks = text_splitter.split_documents(documents)
print(f"chunks are: {chunks[0]}")
print(f"nothing: {chunks}")

print(f"Created {len(chunks)} chunks from {len(documents)} number of documents")
print(f"Content {chunks[0].page_content[:150]}")

chunks are: page_content='Machine learning (ML) and Generative AI (GenAI) are both subsets of Artificial Intelligence (AI), but they have distinct purposes. ML focuses on enabling systems to learn from data and make predictions or decisions, while GenAI aims to generate new, original content like text, images, or music.' metadata={'source': 'data/doc_0.txt'}
nothing: [Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine learning (ML) and Generative AI (GenAI) are both subsets of Artificial Intelligence (AI), but they have distinct purposes. ML focuses on enabling systems to learn from data and make predictions or decisions, while GenAI aims to generate new, original content like text, images, or music.'), Document(metadata={'source': 'data/doc_0.txt'}, page_content='original content like text, images, or music. \nMachine Learning (ML):\nDefinition:\nML is a field of AI that focuses on developing algorithms that allow computers to learn from data without explicit progra

Embedding Models


In [21]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

TypeError: str expected, not NoneType

In [22]:
## OpenAI or Hugging face models can be used to create embeddings

sample_text = "Gen AI is a great skill"
embeddings = OpenAIEmbeddings()


vector=embeddings.embed_query(sample_text)
vector



OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

Initializing the ChromeDB as vector store and storing the chunks in Vector DB

In [25]:
persistent_directory = "./chroma_db"

vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding= OpenAIEmbeddings(),
    persist_directory= persistent_directory,
    collections_name = "rag_collection"
)

print(f"Vector store created with { vectorStore._collection.count() } vectors")
print(f"Persisted to: {persistent_directory}")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
query = "Give me a types of machine learning"

similarity_search_results = vectorStore.similarity_search(query,k=3)
similarity_search_results


Similarity Search with Scores - Advance

In [ ]:
score_results = vectorStore.similarity_search_with_score(query,k=3)
score_results

Context of measuring Similarity Scores

Similarity score is basically depiction of how closely document chunk is related to a query. Distance matrix is calculated using:

Chroma DB utilizes L2 Distance (Euclidean distance), reasons are:

Lower Scores means More similar and closer in vector space
0 score means identical vectors
Mostly range is in between 0-2

Cosine Similarity can also be utilized if configured in Chroma DB:
In this, higher scores mean more similarity
Range varies between -1 to 1 and 1 means identical


Initialize LLM,RAG Chain, Prompt Template, Query the RAG System


In [ ]:
#Chat Models can respond directly
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name = "gpt-3.5-turbo"
)

test = llm.invoke("What are you?")

In [26]:
from langchain.chat_models.base import init_chat_model


llm = init_chat_model("openai:gpt-3.5-turbo")
llm

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

Modern RAG Chain

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

##Convert vector store to retriever (A must step)
retriever = vectorStore.as_retriever(
    search_kwargs = {"k":3} ## Retrieve the top 3 chuks in terms of relevancy 
)

In [28]:
## Creation of prompt templates, it is a must step to feed prompts to LLM

from langchain_core.prompts import ChatPromptTemplate
system_prompt = """ You will be working as an assistant to question-answers tasks.
The retrieved context should be serving as a context to generate answer and maximum 4 lines of answer 
should be given back and if you don't know just say it

Context: {context}
"""

In [30]:
prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You will be working as an assistant to question-answers tasks.\nThe retrieved context should be serving as a context to generate answer and maximum 4 lines of answer \nshould be given back and if you don't know just say it\n\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [ ]:
#Create Stuffed Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm,prompt)